# Hafta 4 — ReAct Döngüsü ve Araç Kullanımı

> [`src/react_agent/`](../src/react_agent/) **için kaynak not defteri.**

## Öğrenme hedefleri

1. ReAct paradigmasını (Yao vd., 2023) bir LLM politikası üzerinde kısmen gözlemlenebilir bir karar süreci olarak biçimselleştirmek.
2. Düşünce → Eylem → Gözlem döngüsünü hiçbir ajan çatısı kullanmadan saf Python ile uygulamak.
3. Bozuk LLM çıktısından toparlanabilen sağlam bir metin protokolü ayrıştırıcısı tasarlamak.
4. Dört kanonik hata türünü açıklamak ve hafifletmek: ayrıştırma hataları, eylem halüsinasyonu, araç istisnaları ve sonsuz döngüler.
5. JSON-Schema'sı ajanın neler yapabileceğinin tek doğruluk kaynağı olan bir araç kayıt defteri kurmak.

## 1. Bir kontrol döngüsü olarak ReAct

Bir ReAct ajanı, bir LLM politikası $\pi_\theta$ ile araçlardan oluşan bir ortam $\mathcal{T} = \{T_1, \dots, T_n\}$ arasındaki, deterministik bir ayrıştırıcının aracılık ettiği kapalı döngü etkileşimdir. $t$ anında ajanın durumu sohbet geçmişi $h_t$'dir. Politika yapılandırılmış bir eylem üretir:

$$
a_t = \pi_\theta(h_t) \in \mathcal{A} = \mathcal{T} \times \mathcal{D} \;\cup\; \{\text{HALT}\},
$$

burada $\mathcal{D}$, (araca özgü) JSON-Schema ile doğrulanan girdi alanıdır. $a_t \neq \text{HALT}$ ise ortam bir gözlem $o_t = T_a(d_a)$ döndürür ve geçmiş güncellenir:

$$
h_{t+1} = h_t \,\Vert\, a_t \,\Vert\, o_t.
$$

Döngü, $a_t = \text{HALT}$ olduğunda (model nihai bir yanıt ürettiğinde) veya katı bir bütçe $T_{\max}$ tükendiğinde sonlanır.

Araç kullanımının tek yolu bu değildir — yerel fonksiyon çağırma API'leri (OpenAI, Anthropic) şemayı çıkarım çağrısının içine gömer. Biz bunun yerine bir metin protokolü kullanıyoruz çünkü (a) şeffaftır, (b) herhangi bir LLM ile çalışır ve (c) uygulamayı bozuk çıktıyı sağlam biçimde ele almaya zorlar — ki pratikte bunu yapmak zorundadır.

## 2. Metin protokolü

Ajan, modelden izleri iki biçimden birinde üretmesini ister:

**Araç çağrısı adımı:**
```
Thought: <serbest biçimli akıl yürütme>
Action: <araç_adı>
Action Input: <aracın input_schema'sına uyan JSON nesnesi>
```

**Son adım:**
```
Thought: <serbest biçimli akıl yürütme>
Final Answer: <serbest biçimli yanıt>
```

Ayrıştırıcı ([`src/react_agent/parser.py`](../src/react_agent/parser.py)) her iki biçimi de tanır ve ajan döngüsünün tükettiği bir `Step` nesnesi üretir. Alan etiketleri (`Thought`, `Action`, `Action Input`, `Final Answer`) protokolün parçası olduğundan İngilizce kalır.

In [ ]:
from src.react_agent.parser import parse_step, ParseError

raw_tool_call = '''Thought: 7 ile 6'nın çarpımına ihtiyacım var.
Action: calculator
Action Input: {"expression": "7 * 6"}'''

step = parse_step(raw_tool_call)
print("is_final:", step.is_final)
print("action:  ", step.action)
print("girdi:   ", step.action_input)


In [ ]:
raw_final = '''Thought: Çarpım 42.
Final Answer: 42'''

step = parse_step(raw_final)
print("is_final:    ", step.is_final)
print("final_answer:", step.final_answer)


In [ ]:
# Ayrıştırıcı JSON etrafındaki kod çitlerini tolere eder — LLM'ler bunları eklemeye bayılır.
forgiving = '''Thought: hesapla.
Action: calculator
Action Input: ```json
{"expression": "2 ** 10"}
```'''

print(parse_step(forgiving).action_input)


In [ ]:
# Ayrıştırma başarısız olduğunda, ajanın toparlanabileceği tipli bir hata üretiriz.
try:
    parse_step("Thought: sadece yüksek sesle düşünüyorum")
except ParseError as e:
    print("ayrıştırma hatası:", e)


## 3. Şemayla doğrulanan çağrılabilirler olarak araçlar

Bir araç, Hafta 1'de MCP için kullandığımız `Tool` nesnesinin aynısıdır — JSON-Schema'sı sözleşmedir.

In [ ]:
from src.react_agent.tools import calculator, read_file

print(calculator.describe())


In [ ]:
# Doğrulayıcı, bozuk argümanları işleyici çalışmadan önce yakalar.
from src.mcp_core.protocol import MCPError
try:
    calculator.invoke({"expression": 123})    # dize değil
except MCPError as exc:
    print("reddedildi:", exc.code, exc.message)


## 4. Ajan döngüsü

Ajanın kendisi 100 satırlık bir `while` döngüsüdür:

In [ ]:
from src.react_agent.agent import ReActAgent
from src.utils.llm_client import Message

class FakeLLM:
    """Hazır ReAct adımlarını sırayla döndüren senaryolu bir LLM."""
    def __init__(self, scripted): self.scripted = list(scripted)
    def complete(self, messages, **kw): return self.scripted.pop(0)

# İki adımlı bir plan: hesapla, sonra yanıtla.
llm = FakeLLM([
    'Thought: 17 * 23 işlemini hesaplayacağım.\nAction: calculator\nAction Input: {"expression": "17 * 23"}',
    'Thought: 17 × 23 = 391.\nFinal Answer: 17 * 23 = 391',
])

agent = ReActAgent(llm=llm, tools=[calculator], max_steps=5)
run = agent.run("17 çarpı 23 kaçtır?")

print("yanıt:         ", run.answer)
print("adımlar:       ", run.steps)
print("halted_reason: ", run.halted_reason)
print("---")
for i, entry in enumerate(run.trace, 1):
    print(f"adım {i}: action={entry.action!r:14s} observation={entry.observation!r}")


## 5. Dört hata türü

Sağlam bir ajan bunların her birini zarifçe ele alır. Bu uygulama da öyle yapar.

### 5a. Ayrıştırma hatası

LLM bozuk çıktı üretirse ajan düzeltici bir mesaj enjekte eder ve durmadan önce `parse_retry_budget` kadar yeniden dener.

In [ ]:
llm = FakeLLM([
    "bu gerekli biçimde değil",                                # anlamsız çıktı
    "Thought: tamam, şimdi ayrıştırılabilir.\nFinal Answer: merhaba",  # düzeltici mesajdan sonra toparlanır
])
agent = ReActAgent(llm=llm, tools=[calculator], max_steps=5, parse_retry_budget=2)
run = agent.run("ping")
print("yanıt:", run.answer, " | halted_reason:", run.halted_reason)


### 5b. Eylem halüsinasyonu

LLM var olmayan bir aracı çağırırsa hata bir Gözlem olarak iletilir. LLM ardından kendini düzeltebilir.

In [ ]:
llm = FakeLLM([
    'Thought: hayalet bir araç dene.\nAction: ghost_tool\nAction Input: {}',
    'Thought: o araç mevcut değil.\nFinal Answer: ghost_tool aracından vazgeçmek zorunda kaldım.',
])
agent = ReActAgent(llm=llm, tools=[calculator], max_steps=3)
run = agent.run("bir araç uydur")
print("adım 1 gözlemi:", run.trace[0].observation[:80], "...")
print("nihai yanıt:   ", run.answer)


### 5c. Araç istisnası

İşleyici bir istisna fırlatırsa istisna bir Gözleme dönüştürülür. Ajan uyum sağlayabilir.

In [ ]:
llm = FakeLLM([
    'Thought: sıfıra böl.\nAction: calculator\nAction Input: {"expression": "1/0"}',
    'Thought: sıfıra bölme hatası. Başka bir şey deneyeceğim.\nFinal Answer: sıfıra bölünemez.',
])
agent = ReActAgent(llm=llm, tools=[calculator], max_steps=3)
run = agent.run("1/0")
print("adım 1 gözlemi:", run.trace[0].observation)
print("nihai yanıt:   ", run.answer)


### 5d. Sonsuz döngüler

Katı bir `max_steps` bütçesi döngüyü sınırlar. Ajan `halted_reason="max_steps"` bildirir; böylece çağıran taraf gerçek bir yanıtı zorla sonlandırmadan ayırt edebilir.

In [ ]:
# Asla Final Answer üretmeyen bir LLM.
looper = 'Thought: henüz değil.\nAction: calculator\nAction Input: {"expression": "1+1"}'
llm = FakeLLM([looper] * 10)
agent = ReActAgent(llm=llm, tools=[calculator], max_steps=3)
run = agent.run("sonsuza dek döngüde kal")
print(run.halted_reason, "—", run.steps, "adımdan sonra")


## 6. Hepsini bir araya getirmek: gerçek bir arXiv araştırma ajanı

Aynı döngü, gerçek araçlarla (arXiv araması + hesap makinesi) çok adımlı araştırma görevlerini çözer. Not defteri canlı çağrıyı çalıştırmaz (bir API anahtarı gerektirir), ancak desen şöyledir:

```python
from src.utils.llm_client import build_client
from src.react_agent.agent import ReActAgent
from src.react_agent.tools import calculator, search_arxiv

agent = ReActAgent(
    llm=build_client("anthropic", "claude-opus-5-5"),
    tools=[calculator, search_arxiv],
    max_steps=10,
)
run = agent.run(
    "HyDE üzerine 2024'ten üç makale bul, başlıklarını listele "
    "ve özetlerinin ortalama sözcük uzunluğunu hesapla."
)
print(run.answer)
```

Model, veri toplamak için `search_arxiv` çağrılarını, toplulaştırmak için `calculator` çağrılarıyla iç içe kullanır, ardından bir Final Answer üretir. Her adım yeni bir LLM çağrısıdır; onları birbirine bağlayan tek şey sohbet geçmişidir.

## 7. Neden ajan çatısı yok

LangChain ve CrewAI gibi çatılar yukarıdaki döngüyü soyutlama katmanlarının arkasına gizler. Bedeli: her çatı güncellemesi güvendiğiniz davranışı bozabilir ve hata ayıklamak sarmalayıcıların içini kazmayı gerektirir.

Bu not defterindeki ajan, ayrıştırıcı, kayıt defteri ve döngü dâhil **~150 satır** koddur. Bir kez yazıldıktan sonra **kararlıdır**: üst akış bağımlılık çalkantısı yok, davranışsal kayma yok. Ödediğiniz bedel, döngüyü sizin yazmanız (ve sahiplenmeniz)dir. Üretimde güvenilir biçimde çalışması gereken sistemler için doğru takas budur.

## 8. Alıştırmalar

1. **İz ayrıntı düzeyi.** `level=2` olduğunda döngü çalışırken her Düşünce / Eylem / Gözlemi yazdıran bir `verbosity` parametresi ekleyin. (İpucu: `structlog.contextvars.bind_contextvars`.)
2. **Kısıtlı kod çözme.** Metin protokolü ayrıştırıcısını, yerel araç kullanımı API'sini kullanan JSON Schema kısıtlı bir kod çözücüyle değiştirin. Ayrıştırma hatası oranını ölçün.
3. **Akış hâlinde Gözlemler.** Bir araç 50 MB'lık bir sonuç üretirse ne olur? Ajanı, gözlemleri ilk $K$ token'a kırpacak ve tam sonucu diske kaydedecek şekilde değiştirin; daha fazlasını getirmek için bir sondan-okuma aracı sunun.
4. **Öz-tutarlılık.** Ajanı $T > 0$ sıcaklığında $K$ kez çalıştırın ve nihai yanıt üzerinde çoğunluk oylaması yapın. 20 soruluk bir matematik kıyaslamasında doğruluğu $K$'ya göre çizdirin.

## 9. Çıkarımlar

- ReAct bir kütüphane değil, bir *kontrol döngüsüdür*. Uygulama tek bir ekrana sığar.
- **Şema**, LLM ile araç arasındaki **sözleşmedir**. Çağırmadan önce doğrulayın.
- Hata türlerini — ayrıştırma, halüsinasyon, istisna, sonsuz döngü — en baştan planlayın. Her birinin belirli, toparlanabilir bir önlemi vardır.
- Yerel fonksiyon çağırma API'leri iyi bir optimizasyondur, ancak ajanınızın ne yaptığını *anlamak* ve *hata ayıklamak* için doğru zemin metin protokolüdür.

➡ Gelecek hafta tek bir ajandan, durum ve bellekle iş birliği yapan özelleşmiş ajanlardan oluşan bir *sisteme* geçiyoruz.